# 04 — what the archive is missing, and why

Every hole the lake knows about is a row or a flag somewhere: trade-id gaps in silver, venue replays, Kraken checksum failures by hour, and the acknowledged offset gaps and chaos windows in `audit.checks`. This notebook reads them all.

In [ ]:
from k2lake import connect, pin
con = connect()
PIN = pin(con)   # every query below reads these snapshot ids, never the moving head
con.sql("""
SELECT 'binance' AS venue, count(*) AS rows, sum(venue_replay::INT) AS replays, sum(seq_gap::INT) AS gaps, sum(missing_before) AS ids_never_received FROM pinned.silver_trades_binance
UNION ALL SELECT 'kraken', count(*), sum(venue_replay::INT), sum(seq_gap::INT), sum(missing_before) FROM pinned.silver_trades_kraken
UNION ALL SELECT 'coinbase', count(*), sum(venue_replay::INT), sum(seq_gap::INT), sum(missing_before) FROM pinned.silver_trades_coinbase
""").show()

When were trades missed? Gaps by hour, per venue — a capture restart, a produce-error drop or a retention eviction each leave a signature.

In [ ]:
con.sql("""
WITH g AS (
  SELECT 'binance' AS venue, date_trunc('hour', exchange_ts) AS h, missing_before FROM pinned.silver_trades_binance WHERE seq_gap
  UNION ALL SELECT 'kraken', date_trunc('hour', exchange_ts), missing_before FROM pinned.silver_trades_kraken WHERE seq_gap
  UNION ALL SELECT 'coinbase', date_trunc('hour', exchange_ts), missing_before FROM pinned.silver_trades_coinbase WHERE seq_gap)
SELECT h, venue, count(*) AS gaps, sum(missing_before) AS ids FROM g GROUP BY 1, 2 ORDER BY 1, 2
""").show(max_rows=40)

Kraken book integrity, per hour: frames whose replayed book hashed to the venue's checksum, failed it, or could not be checked (no snapshot in the archive for that connection).

In [ ]:
con.sql("""
SELECT date_trunc('hour', recv_ts) AS h, sum(CASE WHEN checksum_ok THEN 1 ELSE 0 END) AS verified,
       sum(CASE WHEN checksum_ok = false THEN 1 ELSE 0 END) AS failed, sum(CASE WHEN checksum_ok IS NULL THEN 1 ELSE 0 END) AS unverifiable
FROM pinned.silver_book_kraken GROUP BY 1 ORDER BY 1
""").show(max_rows=48)

And the ledger: what an operator has already acknowledged, and what the nightly audits found.

In [ ]:
con.sql("""
SELECT run_ts, job, check_name, scope, passed, observed, substr(detail, 1, 110) AS detail
FROM pinned.audit_checks WHERE job = 'operator' OR NOT passed ORDER BY run_ts DESC LIMIT 20
""").show(max_width=200)